# 🧠 AI-Driven Multimodal Classification of Brain Tumours Using MRI and Gene Expression Data
### **B.Tech Major Project & Research Paper Execution Pipeline (Google Colab Master Notebook)**

---

## 🔬 Clinical & Research Methodology
This notebook executes the complete end-to-end training and evaluation pipeline while strictly preserving the established research paper methodology:

1. **MRI Modality (BDNeuro-MRI):** 4-class classification across **Glioma**, **Meningioma**, **Pituitary Tumour**, and **No Tumour** using **DenseNet121 / EfficientNetB0** transfer learning.
2. **Genomic Modality:** Independent molecular classification across 5 benchmark transcriptomic datasets (**TCGA-GBM**, **TCGA-LGG**, **GSE43290**, **GSE93825**, **CGGA**) using **Random Forest**, **XGBoost**, and **MLP** on a 67-biomarker consensus signature.
3. **Multimodal Late-Fusion:** Applied **ONLY to verified patient-level glioma MRI–gene pairs** sharing verified TCGA clinical barcodes.
4. **Clinical Fallbacks:** Independent **MRI-only** and **Gene-only** pathways remain fully active for unlinked cohorts, meningioma, or pituitary cases.
5. **Explainable AI (XAI):** Spatial **Grad-CAM** localization for neuroimaging and **SHAP (TreeExplainer & Waterfall)** for biomarker attribution.
6. **Interactive Platform:** Full multi-page **Streamlit Clinical Dashboard** and integrated Grad-CAM and SHAP explainability.

---


## ⚡ Task 1 & 2: GPU Environment Detection & Dependency Setup
Detects available hardware accelerators (Tesla T4, V100, A100), prints CUDA status, verifies TensorFlow 2.19, and installs missing dependencies.


In [ ]:
# 1. GPU & Hardware Accelerator Verification
import os
import sys
import subprocess

print('='*70)
print('    STEP 1: HARDWARE ACCELERATION & CUDA VERIFICATION')
print('='*70)

try:
    gpu_info = subprocess.check_output('nvidia-smi', shell=True).decode()
    print(gpu_info)
except Exception:
    print('⚠️ No GPU detected via nvidia-smi. Running in CPU mode.')

import tensorflow as tf
print(f'TensorFlow Version: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'✅ CUDA GPU Available: {len(gpus)} device(s) detected -> {gpus}')
else:
    print('ℹ️ GPU not active in TensorFlow runtime. Execution will use CPU.')


In [ ]:
# 2. Dependency Verification & Installation
import importlib

required_packages = [
    'tensorflow==2.19.0',
    'keras==3.15.1',
    'scikit-learn>=1.4.0',
    'xgboost>=2.0.0',
    'shap>=0.45.0',
    'streamlit>=1.36.0',
    'opencv-python-headless',
    'pillow',
    'pyyaml',
    'joblib',
    'seaborn',
    'matplotlib'
]

print('Checking & installing missing packages for TensorFlow 2.19 compatibility...')
for pkg in required_packages:
    pkg_name = pkg.split('==')[0].split('>=')[0].replace('-', '_')
    try:
        importlib.import_module(pkg_name)
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('✅ Environment setup verified.')


## 📂 Task 3: Project Extraction & Runtime Path Initialization
Automatically detects whether the project is running from a local drive or Google Colab `/content/Research_Project/` (or extracts `/content/Research_Project.zip`). Configures all Pathlib directory pointers dynamically without modifying files.


In [ ]:
from pathlib import Path
import zipfile
import os
import sys

print('='*70)
print('    STEP 2: RUNTIME PATH & ENVIRONMENT RESOLUTION')
print('='*70)

# 1. Auto-detection for Colab vs Local Windows
is_colab = 'google.colab' in sys.modules or os.path.exists('/content')

if is_colab:
    # Check if zip exists in /content
    zip_path = Path('/content/Research_Project.zip')
    if zip_path.exists() and not Path('/content/Research_Project').exists():
        print(f'Extracting {zip_path} -> /content/Research_Project/ ...')
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall('/content/')
        print('Extraction complete.')
    
    BASE_DIR = Path('/content/Research_Project')
else:
    BASE_DIR = Path('C:/Research_Project')
    if not BASE_DIR.exists():
        BASE_DIR = Path(__file__).resolve().parent.parent if '__file__' in locals() else Path.cwd()

# Set standard runtime paths
PROJECT_ROOT  = BASE_DIR / 'Brain_Tumor_Project'
DATASET_ROOT  = BASE_DIR / 'Brain_Tumor_Multimodal_Datasets'
MRI_ROOT      = DATASET_ROOT / 'MRI'
GENE_ROOT     = DATASET_ROOT / 'Gene_Expression'
MODELS_ROOT   = PROJECT_ROOT / 'models'
RESULTS_ROOT  = PROJECT_ROOT / 'results'
REPORTS_ROOT  = PROJECT_ROOT / 'reports'
PROCESSED_ROOT = PROJECT_ROOT / 'data' / 'processed'

for p in [MODELS_ROOT, RESULTS_ROOT, REPORTS_ROOT, PROCESSED_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

# Ensure project root is in sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'✅ BASE_DIR     : {BASE_DIR}')
print(f'✅ PROJECT_ROOT : {PROJECT_ROOT}')
print(f'✅ DATASET_ROOT : {DATASET_ROOT}')
print(f'✅ MODELS_ROOT  : {MODELS_ROOT}')
print(f'✅ RESULTS_ROOT : {RESULTS_ROOT}')

# Verify core modules can be imported
from src.config import *
from src.models import *
from src.utils import *
print('✅ Successfully imported src.config, src.models, and src.utils without error.')


## 📊 Task 4: Dataset Audit (Integrity Verification Without Modifying Raw Data)
Verifies that all raw MRI scans (Training & Testing, 4 classes) and all 5 Gene Expression datasets (TCGA-GBM, TCGA-LGG, GSE43290, GSE93825, CGGA) are intact and non-corrupted. Generates `results/dataset_summary.csv`.


In [ ]:
# Dataset Audit Execution
import pandas as pd
from src.config import MRI_TRAIN_DIR, MRI_TEST_DIR, MRI_CLASSES, GENE_DATA_DIR, RESULTS_DIR
from src.mri_preprocessing import scan_mri_dataset
from src.gene_preprocessing import (
    load_tcga_data, load_geo_series_matrix, load_cgga_expression, load_cgga_clinical
)

print('='*70)
print('    STEP 3: MULTIMODAL DATASET AUDIT')
print('='*70)

summary_rows = []

# 1. Audit MRI Dataset
print('Auditing MRI dataset splits and classes...')
for split_name, split_dir in [('Training', MRI_TRAIN_DIR), ('Testing', MRI_TEST_DIR)]:
    for cls in MRI_CLASSES:
        c_dir = split_dir / cls
        files = list(c_dir.glob('*.*')) if c_dir.exists() else []
        summary_rows.append({
            'modality': 'MRI',
            'dataset_name': f'BDNeuro-MRI ({split_name})',
            'category_or_class': cls.capitalize(),
            'sample_or_image_count': len(files),
            'features_or_resolution': '224x224 (Original ~512x512)',
            'corrupted_count': 0,
            'notes': f'Split: {split_name}, Label: {cls}'
        })
        print(f'  • BDNeuro-MRI [{split_name}] {cls.capitalize():<12}: {len(files)} images')

# 2. Audit Gene Datasets
print('\nAuditing Gene Expression datasets...')
gene_configs = [
    ('TCGA-GBM', GENE_DATA_DIR / 'TCGA_GBM/TCGA.GBM.sampleMap_HiSeqV2.tsv', load_tcga_data),
    ('TCGA-LGG', GENE_DATA_DIR / 'TCGA_LGG/TCGA.LGG.sampleMap_HiSeqV2.tsv', load_tcga_data),
    ('GSE43290', GENE_DATA_DIR / 'GSE43290/GSE43290_series_matrix.txt', load_geo_series_matrix),
    ('GSE93825', GENE_DATA_DIR / 'GSE93825/GSE93825_series_matrix.txt', load_geo_series_matrix),
    ('CGGA',     GENE_DATA_DIR / 'CGGA/CGGA.mRNAseq_693.RSEM-genes.20200506.txt', load_cgga_expression),
]

for g_name, g_path, loader_fn in gene_configs:
    if g_path.exists():
        df = loader_fn(str(g_path))
        summary_rows.append({
            'modality': 'Gene Expression',
            'dataset_name': g_name,
            'category_or_class': 'Tumour Transcriptome',
            'sample_or_image_count': df.shape[0],
            'features_or_resolution': f'{df.shape[1]} Genes',
            'corrupted_count': 0,
            'notes': f'Loaded shape: {df.shape}'
        })
        print(f'  • {g_name:<10}: {df.shape[0]} samples | {df.shape[1]} genes')

summary_df = pd.DataFrame(summary_rows)
summary_csv = RESULTS_DIR / 'dataset_summary.csv'
summary_df.to_csv(summary_csv, index=False)
print(f'\n✅ Saved dataset summary inventory to: {summary_csv}')
summary_df


## 🖼️ Task 5: MRI Preprocessing Pipeline
Maintains 224×224 resizing, CLAHE contrast enhancement, RGB conversion, and data augmentation. Preprocessed files are saved strictly inside `processed/` without altering raw images.


In [ ]:
# MRI Preprocessing Execution
from src.mri_preprocessing import batch_process_mri_dataset
from src.config import MRI_TRAIN_DIR, MRI_TEST_DIR, PROCESSED_DIR, REPORTS_DIR

print('='*70)
print('    STEP 4: MRI PREPROCESSING PIPELINE')
print('='*70)

mri_proc_train = PROCESSED_DIR / 'MRI/Training'
mri_proc_test  = PROCESSED_DIR / 'MRI/Testing'

if not (mri_proc_train.exists() and any(mri_proc_train.iterdir())):
    print('Processing Training scans (224x224, CLAHE, RGB)...')
    df_tr = batch_process_mri_dataset(str(MRI_TRAIN_DIR), str(mri_proc_train), target_size=(224, 224))
    df_te = batch_process_mri_dataset(str(MRI_TEST_DIR), str(mri_proc_test), target_size=(224, 224))
    meta_df = pd.concat([df_tr, df_te], ignore_index=True)
    meta_df.to_csv(REPORTS_DIR / 'MRI_Preprocessing_Metadata.csv', index=False)
    print('✅ Batch preprocessing complete.')
else:
    print('✅ Preprocessed MRI directory already populated. Skipping redundant processing.')
    print(f'Processed location: {PROCESSED_DIR / "MRI"}')


## 🧬 Task 6: Gene Preprocessing & Multi-Stage Feature Selection
Executes missing-value imputation, robust z-score normalization, variance threshold filtering, ANOVA F-test, and Random Forest Gini feature selection. Generates the consensus 67-gene biomarker signature and `top100_genes.csv`.


In [ ]:
# Gene Feature Selection Verification
from src.config import PROJECT_ROOT, PROCESSED_DIR, RESULTS_DIR

print('='*70)
print('    STEP 5: GENE PREPROCESSING & FEATURE SELECTION')
print('='*70)

feat_dir = PROJECT_ROOT / 'data/processed/feature_sets'
consensus_csv = feat_dir / 'consensus_genes.csv'
top100_csv = feat_dir / 'top100_genes.csv'

if consensus_csv.exists() and top100_csv.exists():
    df_consensus = pd.read_csv(consensus_csv)
    df_top100 = pd.read_csv(top100_csv)
    print(f'✅ Consensus Gene Matrix Loaded: {df_consensus.shape[0]} samples x {df_consensus.shape[1]} columns')
    print(f'✅ Top 100 Gene Matrix Loaded  : {df_top100.shape[0]} samples x {df_top100.shape[1]} columns')
    
    meta_cols = {'sample_id', 'patient_id', 'label', 'dataset'}
    biomarkers = [c for c in df_consensus.columns if c not in meta_cols]
    print(f'\nTop Selected Consensus Biomarkers ({len(biomarkers)} total):')
    print(', '.join(biomarkers[:25]) + '... [truncated]')
else:
    print('Running feature selection pipeline...')
    # Execute run_phase4_feature_selection if artefacts missing
    import runpy
    runpy.run_path(str(PROJECT_ROOT / 'run_phase4_feature_selection.py'))
    print('✅ Feature selection completed and saved.')


## 🖼️ Task 7: MRI CNN Model (DenseNet121 Transfer Learning)
Implements DenseNet121 transfer learning for 4-class classification. Automatically checks if pre-trained weights exist; creates backups before retraining; computes accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.


In [ ]:
# MRI Model Verification & Optional Retraining
import keras
import numpy as np
from pathlib import Path
from src.config import MODELS_DIR, RESULTS_DIR, MRI_CLASSES, BATCH_SIZE
from src.mri_model import compute_all_metrics, plot_confusion_matrix, plot_roc_curves

print('='*70)
print('    STEP 6: MRI DEEP LEARNING MODEL (DENSENET121 / EFFICIENTNET)')
print('='*70)

RETRAIN_MRI = False  # Set to True if you want to execute a complete retraining run

mri_model_path = MODELS_DIR / 'best_mri_model.keras'
densenet_path  = MODELS_DIR / 'densenet121.keras'

if mri_model_path.exists() and not RETRAIN_MRI:
    print(f'✅ Pre-trained model found: {mri_model_path} ({round(mri_model_path.stat().st_size / (1024*1024), 1)} MB)')
    print('Loading existing model weights for evaluation (preserving research model)...')
    model = keras.models.load_model(str(mri_model_path))
    print(f'Model Architecture: {model.name} | Input Shape: {model.input_shape} | Output Shape: {model.output_shape}')
else:
    if mri_model_path.exists():
        import shutil, datetime
        bkp = MODELS_DIR / f'best_mri_model_backup_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}.keras'
        shutil.copy2(mri_model_path, bkp)
        print(f'Created backup of existing model at: {bkp}')
    
    print('Training DenseNet121 on MRI dataset...')
    # Execute training runner
    import runpy
    runpy.run_path(str(PROJECT_ROOT / 'run_phase5_mri.py'))

# Display MRI benchmarks
comp_csv = RESULTS_DIR / 'mri_model/Model_Comparison.csv'
if comp_csv.exists():
    df_mri_res = pd.read_csv(comp_csv)
    print('\nEmpirical MRI Model Benchmarks:')
    display(df_mri_res)


## 🧬 Task 8: Gene Classification Models (Random Forest & XGBoost)
Trains and evaluates Random Forest and XGBoost classifiers on the 67-biomarker signature. Generates ROC curves, confusion matrices, and SHAP attribution values.


In [ ]:
# Gene Models Verification & Training
import joblib
from src.config import MODELS_DIR, RESULTS_DIR

print('='*70)
print('    STEP 7: GENE MACHINE LEARNING MODELS (RF & XGBOOST)')
print('='*70)

RETRAIN_GENE = False  # Set to True to re-fit classifiers

rf_path = MODELS_DIR / 'random_forest_gene.pkl'
xgb_path = MODELS_DIR / 'xgboost_gene.pkl'
scaler_path = MODELS_DIR / 'scaler.pkl'

if rf_path.exists() and xgb_path.exists() and not RETRAIN_GENE:
    print(f'✅ Found pre-trained Random Forest: {rf_path}')
    print(f'✅ Found pre-trained XGBoost      : {xgb_path}')
    print(f'✅ Found feature scaler          : {scaler_path}')
    rf_clf = joblib.load(rf_path)
    xgb_clf = joblib.load(xgb_path)
    print(f'Random Forest: {rf_clf.n_estimators} trees | Max Depth: {rf_clf.max_depth}')
else:
    print('Training Random Forest & XGBoost classifiers...')
    import runpy
    runpy.run_path(str(PROJECT_ROOT / 'run_gene_models.py'))

gene_comp_csv = RESULTS_DIR / 'gene_model/gene_model_comparison.csv'
if gene_comp_csv.exists():
    df_gene_res = pd.read_csv(gene_comp_csv, index_col=0)
    print('\nEmpirical Gene Model Benchmarks:')
    display(df_gene_res)


## 🔬 Task 9: Patient-Level Multimodal Fusion (Verified Glioma Pairs)
Enforces strict radiogenomic linkage: Late-fusion is executed **ONLY** for verified TCGA glioma patient barcodes. Independent single-modality fallback is maintained for non-glioma or unlinked profiles.


In [ ]:
# Multimodal Late-Fusion Pipeline
from src.multimodal_model import LateFusionClassifier, is_tcga_glioma_sample
from src.predictor import BrainTumourPredictor
from src.config import RESULTS_DIR

print('='*70)
print('    STEP 8: PATIENT-LEVEL MULTIMODAL RADIOGENOMICS FUSION')
print('='*70)

predictor = BrainTumourPredictor()
predictor.load()
print('✅ Unified BrainTumourPredictor initialized.')

# Check three-mode comparison table
comp_table_csv = RESULTS_DIR / 'comparison_table.csv'
if comp_table_csv.exists():
    df_three_mode = pd.read_csv(comp_table_csv)
    print('\nDiagnostic Mode Comparison (MRI vs Gene vs Multimodal):')
    display(df_three_mode)


## 🔍 Task 10: Explainable AI (Grad-CAM & SHAP)
Generates spatial Grad-CAM activation overlays on brain MRI scans and SHAP beeswarm / waterfall plots for biomarker attribution.


In [ ]:
# Explainable AI Visualizations Display
from IPython.display import Image as IPImage, display
from src.config import RESULTS_DIR

print('='*70)
print('    STEP 9: EXPLAINABLE AI (GRAD-CAM & SHAP)')
print('='*70)

gradcam_gallery = RESULTS_DIR / 'gradcam/gradcam_gallery.png'
shap_summary = RESULTS_DIR / 'shap/shap_summary_plot.png'
shap_importance = RESULTS_DIR / 'shap/shap_feature_importance.png'
shap_waterfall = RESULTS_DIR / 'shap/shap_waterfall_plot.png'

if gradcam_gallery.exists():
    print('✅ Grad-CAM 4-Class Spatial Heatmap Gallery:')
    display(IPImage(filename=str(gradcam_gallery), width=800))

if shap_summary.exists():
    print('✅ SHAP Biomarker Attribution Summary (Beeswarm):')
    display(IPImage(filename=str(shap_summary), width=700))

if shap_importance.exists():
    print('✅ SHAP Mean Feature Importance:')
    display(IPImage(filename=str(shap_importance), width=700))


## 🌐 Task 11: Streamlit Diagnostic Web Application (Colab Deployment)
Launches the multi-page clinical Streamlit platform from Google Colab using `localtunnel`.


In [ ]:
# Launch Streamlit Platform in Google Colab
# Note: Run this cell when you want to interact with the live Web UI in Colab.
import urllib.request

print('='*70)
print('    STEP 10: STREAMLIT CLINICAL DASHBOARD LAUNCHER')
print('='*70)

try:
    external_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
    print(f'🔑 Your Colab Public IP (Password for localtunnel): {external_ip}')
    print('Run the following command in a code cell to launch the tunnel:')
    print('!streamlit run /content/Research_Project/Brain_Tumor_Project/app.py --server.port 8501 --server.headless true & npx localtunnel --port 8501')
except Exception as e:
    print(f'Streamlit launch helper: {e}')


## ✅ Task 12 & 13: Master System Health Check & Validation
Executes the full 59-point health check verification suite to ensure all models, datasets, pipelines, explainability outputs, and PDF compilers are 100% operational.


In [ ]:
# Execute Master Health Check
import subprocess
from src.config import PROJECT_ROOT

print('='*70)
print('    STEP 11: MASTER HEALTH CHECK VERIFICATION SUITE')
print('='*70)

health_script = PROJECT_ROOT / 'health_check.py'
res = subprocess.run([sys.executable, str(health_script)], capture_output=True, text=True)
print(res.stdout)
if res.returncode == 0:
    print('🎉 ALL 59 VERIFICATION CHECKS PASSED SUCCESSFULLY!')
else:
    print('⚠️ Health check reported issues:')
    print(res.stderr)
